# HiC-ECC | Module 1: Enhance (DeepHiC)
Convert HiC-Pro matrices → DeepHiC enhancement → .cool files.

## Config
**Edit only this cell.**

In [1]:
import yaml, re

with open('../../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

TISSUES      = cfg['samples']
GENOME       = cfg['genome']
SPECIES      = cfg['species']
RESOLUTION   = cfg['resolution']
RES_STR      = str(RESOLUTION)
CHROM_SIZES  = cfg['chrom_sizes']
SAMPLE_DIR   = cfg['hicpro_dir']
DEEPHIC_PY   = cfg['enhancement']['deephic_path']
CHECKPOINT   = cfg['enhancement']['checkpoint']
CHUNK        = cfg['enhancement']['chunk']
STRIDE       = cfg['enhancement']['stride']
BOUND        = cfg['enhancement']['bound']
LRC          = cfg['enhancement']['lrc']
DEEPHIC_ROOT = f"{cfg['output_dir']}/DeepHiC/{GENOME}"
COOL_DIR     = f"{cfg['output_dir']}/cool_files/{RESOLUTION}"

# Patch all_parser.py root_dir to match DEEPHIC_ROOT
with open(f'{DEEPHIC_PY}/all_parser.py', 'r') as f:
    _content = f.read()
_content = re.sub(r"root_dir = '.*'", f"root_dir = '{DEEPHIC_ROOT}/'", _content)
with open(f'{DEEPHIC_PY}/all_parser.py', 'w') as f:
    f.write(_content)
print(f'Patched all_parser.py: root_dir = {DEEPHIC_ROOT}/')

Patched all_parser.py: root_dir = /cluster/projects/epigenomics/BACKUP_31032025/EpigenomeLab/Aminnn/Hi-C/Mouse/final_results/enhanced_data/DeepHiC/mm10/


## Setup

In [2]:
import os, subprocess, re
import numpy as np

os.makedirs(f'{DEEPHIC_ROOT}/mat', exist_ok=True)
print('Ready.')

Ready.


# Section 1 - HiC-Pro → DeepHiC → predict

In [3]:
for tissue in TISSUES:
    tissue_dir = f'{SAMPLE_DIR}/{tissue}/hic_results/matrix/{tissue}/raw/{RESOLUTION}'
    predict_dir = f'{DEEPHIC_ROOT}/predict/{tissue}_{RESOLUTION}/sr16'

    if os.path.isdir(predict_dir) and len(os.listdir(predict_dir)) > 0:
        print(f'[{tissue}] prediction exists, skipping')
        continue

    print(f'\n[{tissue}] hicpro2deephic')
    subprocess.run([
        'python', f'{DEEPHIC_PY}/hicpro2deephic.py',
        '--bed', f'{tissue_dir}/{tissue}_{RES_STR}_abs.bed',
        '--mat', f'{tissue_dir}/{tissue}_{RES_STR}.matrix',
        '-r', RES_STR,
        '-o', f'{DEEPHIC_ROOT}/mat/{tissue}_{RES_STR}',
    ], check=True)

    print(f'[{tissue}] data_generate')
    subprocess.run([
        'python', f'{DEEPHIC_PY}/data_generate.py',
        '-hr', RES_STR, '-lr', RES_STR,
        '-lrc', str(LRC), '-s', SPECIES,
        '-chunk', str(CHUNK), '-stride', str(STRIDE),
        '-bound', str(BOUND), '-scale', '1',
        '-c', f'{tissue}_{RES_STR}',
    ], check=True, cwd=DEEPHIC_ROOT)

    print(f'[{tissue}] data_predict')
    subprocess.run([
        'python', f'{DEEPHIC_PY}/data_predict.py',
        '-lr', RES_STR,
        '-ckpt', CHECKPOINT,
        '-c', f'{tissue}_{RES_STR}',
    ], check=True, cwd=DEEPHIC_ROOT)

    print(f'[{tissue}] done')

print('\nAll tissues enhanced.')

[Spleen] prediction exists, skipping
[Kidney] prediction exists, skipping
[Liver] prediction exists, skipping
[Lung] prediction exists, skipping
[Pancreas] prediction exists, skipping

[Large_Intestine] hicpro2deephic


Saving 10000 files to /cluster/projects/epigenomics/BACKUP_31032025/EpigenomeLab/Aminnn/Hi-C/Mouse/final_results/enhanced_data/DeepHiC/mm10/mat/Large_Intestine_10000/chr1_10000.npz
Saving 10000 files to /cluster/projects/epigenomics/BACKUP_31032025/EpigenomeLab/Aminnn/Hi-C/Mouse/final_results/enhanced_data/DeepHiC/mm10/mat/Large_Intestine_10000/chr10_10000.npz
Saving 10000 files to /cluster/projects/epigenomics/BACKUP_31032025/EpigenomeLab/Aminnn/Hi-C/Mouse/final_results/enhanced_data/DeepHiC/mm10/mat/Large_Intestine_10000/chr11_10000.npz
Saving 10000 files to /cluster/projects/epigenomics/BACKUP_31032025/EpigenomeLab/Aminnn/Hi-C/Mouse/final_results/enhanced_data/DeepHiC/mm10/mat/Large_Intestine_10000/chr12_10000.npz
Saving 10000 files to /cluster/projects/epigenomics/BACKUP_31032025/EpigenomeLab/Aminnn/Hi-C/Mouse/final_results/enhanced_data/DeepHiC/mm10/mat/Large_Intestine_10000/chr13_10000.npz
Saving 10000 files to /cluster/projects/epigenomics/BACKUP_31032025/EpigenomeLab/Aminnn/Hi-

########## Large_Intestine_10000 ##########
Going to read 10000 and 10000 data, then deviding matrices with nonpool
23
Start Time: 1777614126.2122056


[Chr16] Deviding HiC matrix (9497x9497) into 8836 samples with chunk=100, stride=100, bound=9999
[Chr19] Deviding HiC matrix (5813x5813) into 3364 samples with chunk=100, stride=100, bound=9999
[Chr15] Deviding HiC matrix (10069x10069) into 10000 samples with chunk=100, stride=100, bound=9999
[Chr17] Deviding HiC matrix (9174x9174) into 8281 samples with chunk=100, stride=100, bound=9999
[Chr18] Deviding HiC matrix (8745x8745) into 7569 samples with chunk=100, stride=100, bound=9999
[Chr11] Deviding HiC matrix (11872x11872) into 13582 samples with chunk=100, stride=100, bound=9999
[Chr13] Deviding HiC matrix (11712x11712) into 13383 samples with chunk=100, stride=100, bound=9999
[Chr12] Deviding HiC matrix (11680x11680) into 13184 samples with chunk=100, stride=100, bound=9999
[Chr14] Deviding HiC matrix (12047x12047) into 13980 samples with chunk=100, stride=100, bound=9999
[Chr9] Deviding HiC matrix (12121x12121) into 14179 samples with chunk=100, stride=100, bound=9999
[Chr10] Devid

Start a multiprocess pool with processes = 23 for generating DeepHiC data
All DeepHiC data generated. Running cost is 5.7 min.
Saving file: /cluster/projects/epigenomics/BACKUP_31032025/EpigenomeLab/Aminnn/Hi-C/Mouse/final_results/enhanced_data/DeepHiC/mm10/data/deephic_1000010000_c100_s100_b9999_nonpool_large_intestine_10000.npz


...

DeepHiC Predicting: 100%|██████████| 4820/4820 [5:33:04<00:00,  4.15s/it]


Reconstructing:  data contain [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 'X'] chromosomes


Spreading a (5812, 5812) shaped matrix to (6144, 6144) shaped!
Saving file: /cluster/projects/epigenomics/BACKUP_31032025/EpigenomeLab/Aminnn/Hi-C/Mouse/final_results/enhanced_data/DeepHiC/mm10/predict/Brain_10000/sr16/predict_chr19_10000.npz
Spreading a (8740, 8740) shaped matrix to (9071, 9071) shaped!
Saving file: /cluster/projects/epigenomics/BACKUP_31032025/EpigenomeLab/Aminnn/Hi-C/Mouse/final_results/enhanced_data/DeepHiC/mm10/predict/Brain_10000/sr16/predict_chr18_10000.npz
Spreading a (9494, 9494) shaped matrix to (9821, 9821) shaped!
Saving file: /cluster/projects/epigenomics/BACKUP_31032025/EpigenomeLab/Aminnn/Hi-C/Mouse/final_results/enhanced_data/DeepHiC/mm10/predict/Brain_10000/sr16/predict_chr16_10000.npz
Spreading a (9170, 9170) shaped matrix to (9499, 9499) shaped!
Saving file: /cluster/projects/epigenomics/BACKUP_31032025/EpigenomeLab/Aminnn/Hi-C/Mouse/final_results/enhanced_data/DeepHiC/mm10/predict/Brain_10000/sr16/predict_chr17_10000.npz
Spreading a (10068, 10068) s

Start a multiprocess pool with process_num = 23 for saving predicted data
dict_keys([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 'X'])
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
X
All data saved. Running cost is 339.0 min.


[Brain] done

All tissues enhanced.


# Section 2 - Convert predictions to .cool

In [4]:
RES_KB      = RESOLUTION // 1000
PREDICT_DIR = f'{DEEPHIC_ROOT}/predict'

for tissue in TISSUES:
    print(f'\n[{tissue}] building bg2')
    tissue_cool_dir = f'{COOL_DIR}/{tissue}'
    os.makedirs(tissue_cool_dir, exist_ok=True)

    bg2  = f'{tissue_cool_dir}/{tissue}_deephic_{RES_KB}kb.bg2'
    cool = f'{tissue_cool_dir}/{tissue}_deephic.{RES_KB}kb.cool'
    sr_dir = f'{PREDICT_DIR}/{tissue}_{RESOLUTION}/sr16'

    if os.path.isfile(cool) and os.path.getsize(cool) > 0:
        print(f'[{tissue}] cool file exists, skipping')
        continue

    # npz → bg2
    with open(bg2, 'w') as out:
        for fname in sorted(os.listdir(sr_dir)):
            if not fname.endswith(f'_{RESOLUTION}.npz'):
                continue
            chrom = fname.replace('predict_', '').replace(f'_{RESOLUTION}.npz', '')
            matrix = np.load(os.path.join(sr_dir, fname))['deephic']
            for i in range(matrix.shape[0]):
                for j in range(i, matrix.shape[0]):
                    if matrix[i, j] > 0:
                        out.write(f'{chrom}\t{i*RESOLUTION}\t{(i+1)*RESOLUTION}\t'
                                  f'{chrom}\t{j*RESOLUTION}\t{(j+1)*RESOLUTION}\t{matrix[i,j]}\n')

    # bg2 → cool
    print(f'[{tissue}] cooler load')
    subprocess.run([
        'cooler', 'load', '-f', 'bg2',
        f'{CHROM_SIZES}:{RESOLUTION}',
        bg2, cool,
        '--count-as-float', '--input-copy-status', 'duplex',
    ], check=True)

    print(f'[{tissue}] cooler balance')
    subprocess.run(['cooler', 'balance', cool, '--force'], check=True)

    print(f'[{tissue}] done → {cool}')

print('\nAll cool files ready.')


[Spleen] building bg2


[Spleen] cooler load


INFO:cooler.cli.load:fields: {'chrom1': 0, 'start1': 1, 'end1': 2, 'chrom2': 3, 'start2': 4, 'end2': 5, 'count': 6}
INFO:cooler.cli.load:dtypes: {'chrom1': <class 'str'>, 'start1': <class 'int'>, 'end1': <class 'int'>, 'chrom2': <class 'str'>, 'start2': <class 'int'>, 'end2': <class 'int'>, 'count': <class 'numpy.float64'>}
INFO:cooler.cli.load:symmetric-upper: True


INFO:cooler.create:Writing chunk 0: /cluster/projects/epigenomics/BACKUP_31032025/EpigenomeLab/Aminnn/Hi-C/Mouse/final_results/enhanced_data/cool_files/10000/Spleen/tmprv_x6wd3.multi.cool::0


INFO:cooler.create:Creating cooler at "/cluster/projects/epigenomics/BACKUP_31032025/EpigenomeLab/Aminnn/Hi-C/Mouse/final_results/enhanced_data/cool_files/10000/Spleen/tmprv_x6wd3.multi.cool::/0"
INFO:cooler.create:Writing chroms
INFO:cooler.create:Writing bins
INFO:cooler.create:Writing pixels


...

INFO:cooler.balance:variance is 39584.44339288949


INFO:cooler.balance:variance is 76033.09004756199


INFO:cooler.balance:variance is 43514.135134834134


INFO:cooler.balance:variance is 83581.17157819957
ERROR:cooler.cli.balance:Iteration limit reached without convergence
ERROR:cooler.cli.balance:Storing final result. Check log to assess convergence.


[Brain] done → /cluster/projects/epigenomics/BACKUP_31032025/EpigenomeLab/Aminnn/Hi-C/Mouse/final_results/enhanced_data/cool_files/10000/Brain/Brain_deephic.10kb.cool

All cool files ready.
